
# End‑to‑End Bug Severity Prediction Project

## 1. Project Overview
This notebook implements a complete, reproducible pipeline that predicts bug severity from textual and categorical metadata. It covers:
1. Data Loading & EDA
2. Outlier Detection (IQR)
3. Correlation Analysis
4. Feature Engineering
5. Chi‑Square feature selection
6. Model training (Logistic Regression, Random Forest, XGBoost, Naive Bayes)
7. Hyper‑parameter tuning for XGBoost
8. Model selection based on macro‑F1
9. Final pipeline serialization
10. Feature importance visualization



## 2. Data Loading & Initial Exploration


In [ ]:

data_path = 'Data/bug_dataset_50k.csv'
df = pd.read_csv(data_path)
print('Dataset shape:', df.shape)
df.head()



### 2.1 Re‑inject deterministic severity logic (if needed)


In [ ]:

def assign_logic(row):
    text = f"{row.get('title','')} {row.get('description','')}".lower()
    if any(w in text for w in ['crash', 'security', 'leak', 'fatal', 'vulnerability']):
        return 'Critical'
    if 'production' in str(row.get('environment','')).lower() or any(w in text for w in ['api', 'database', 'timeout']):
        return 'High'
    if 'staging' in str(row.get('environment','')).lower() or 'backend' in str(row.get('developer_role','')).lower():
        return 'Medium'
    return 'Low'

df['severity'] = df.apply(assign_logic, axis=1)



## 3. Outlier Detection (IQR) on word count


In [ ]:

# Create combined text and word count
df['title'] = df['title'].fillna('')
df['description'] = df['description'].fillna('')
df['full_text'] = (df['title'] + ' ' + df['description']).str.lower()
df['desc_word_count'] = df['description'].apply(lambda x: len(str(x).split()))

Q1 = df['desc_word_count'].quantile(0.25)
Q3 = df['desc_word_count'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
orig_len = len(df)
df = df[(df['desc_word_count'] >= lower) & (df['desc_word_count'] <= upper)]
print(f'Removed {orig_len - len(df)} outlier rows based on word count')



## 4. Correlation Analysis (numeric features)


In [ ]:

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.savefig('correlation_matrix.png')
plt.show()



## 5. Feature Engineering


In [ ]:

# Drop identifiers & unused columns
drop_cols = ['bug_id', 'created_at', 'title', 'description', 'suggested_fix', 'explanation', 'root_cause']
df_ml = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Target encoding
le_target = LabelEncoder()
y = le_target.fit_transform(df_ml['severity'])
X = df_ml.drop(columns=['severity'])
print('Class mapping:', dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))



## 6. Preprocessing (ColumnTransformer) + Chi‑Square feature selection


In [ ]:

text_feat = 'full_text'
numeric_feats = ['error_code', 'desc_word_count']
categorical_feats = ['bug_category', 'bug_domain', 'tech_stack', 'environment', 'developer_role']

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
                         ('scaler', MinMaxScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                             ('onehot', OneHotEncoder(handle_unknown='ignore'))])
text_pipe = Pipeline([('tfidf', TfidfVectorizer(max_features=500, stop_words='english'))])

preprocessor = ColumnTransformer([('num', numeric_pipe, numeric_feats),
                               ('cat', categorical_pipe, categorical_feats),
                               ('text', text_pipe, text_feat)])

chi_selector = SelectKBest(score_func=chi2, k=2000)



## 7. Train/Test Split


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



## 8. Model Building & Baseline Comparison


In [ ]:

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    'Naive Bayes': MultinomialNB()
}

baseline_results = {}
for name, model in models.items():
    print(f'Training {name}...')
    pipe = Pipeline([('preprocessor', preprocessor), ('chi2', chi_selector), ('clf', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    baseline_results[name] = {
        'pipeline': pipe, 
        'Accuracy': accuracy_score(y_test, preds), 
        'Macro_F1': f1_score(y_test, preds, average='macro'), 
        'Preds': preds
    }



## 9. Hyperparameter Tuning (XGBoost)


In [ ]:

xgb_pipe = Pipeline([('preprocessor', preprocessor), ('chi2', chi_selector), ('clf', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42))])
param_grid = {'clf__n_estimators': [50, 100], 'clf__max_depth': [3, 5], 'clf__learning_rate': [0.1, 0.2]}
search = RandomizedSearchCV(xgb_pipe, param_grid, n_iter=3, scoring='f1_macro', cv=2, random_state=42, n_jobs=-1)
search.fit(X_train, y_train)



## 10. Model Selection & Artifact Generation


In [ ]:

best_xgb = search.best_estimator_
preds_xgb = best_xgb.predict(X_test)
baseline_results['XGBoost (Tuned)'] = {
    'pipeline': best_xgb, 
    'Accuracy': accuracy_score(y_test, preds_xgb),
    'Macro_F1': f1_score(y_test, preds_xgb, average='macro'), 
    'Preds': preds_xgb
}

best_name = max(baseline_results, key=lambda k: baseline_results[k]['Macro_F1'])
best_model = baseline_results[best_name]['pipeline']

# Chi-Square Stats
X_train_pre = preprocessor.fit_transform(X_train, y_train)
chi_scores, chi_pvals = chi2(X_train_pre, y_train)
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
tfidf_feat = preprocessor.named_transformers_['text'].named_steps['tfidf'].get_feature_names_out()
all_features = list(numeric_feats) + list(ohe.get_feature_names_out()) + list(tfidf_feat)
chi_df = pd.DataFrame({'feature': all_features, 'chi2_score': chi_scores, 'p_value': chi_pvals})
chi_df.to_csv('chi2_feature_stats.csv', index=False)

# Save
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/best_pipeline.pkl')
joblib.dump(le_target, 'models/target_encoder.pkl')
